# Fine-Tuning Non-LLM Models with HuggingFace Trainer

Not all fine-tuning involves language models. The HuggingFace `Trainer` API
works across the entire `transformers` library — BERT, ViT, Whisper, etc.

This notebook covers two common non-LLM tasks:

| Section | Task | Model | Dataset |
|---------|------|-------|---------|
| **A** | Text Classification | `distilbert-base-uncased` | Inline sentiment examples |
| **B** | Image Classification | `google/vit-base-patch16-224-in21k` | `beans` (plant disease, 3 classes) |

**Key difference from LLM fine-tuning:** we typically do *full* fine-tuning
on small encoders (BERT-base is 110M params, ViT-base is 86M) rather than
LoRA. LoRA is possible but rarely necessary at this scale.

## Environment Setup

```bash
cd projects/huggingface_trl
uv sync --no-install-workspace
uv run ipython kernel install --user --env VIRTUAL_ENV ../../.venv --name=huggingface_trl
```

Select the **`huggingface_trl`** kernel in JupyterLab.

In [ ]:
import sys, torch, transformers, datasets, evaluate as ev
print(f'Python    {sys.version}')
print(f'PyTorch   {torch.__version__}  CUDA: {torch.cuda.is_available()}')
print(f'transformers {transformers.__version__}  datasets {datasets.__version__}')
print(f'evaluate  {ev.__version__}')
if torch.cuda.is_available():
    print(f'GPU       {torch.cuda.get_device_name(0)}')

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('../../../').resolve()))
from dotenv_config import dot_env_settings
if dot_env_settings.HF_TOKEN:
    os.environ['HF_TOKEN'] = dot_env_settings.HF_TOKEN
    print('HF_TOKEN loaded.')
else:
    print('WARNING: No HF_TOKEN — public models and datasets only.')

---
## Section A — Text Classification with DistilBERT

**Task:** binary sentiment classification (positive / negative)

The workflow for any encoder-based text classification task is:
1. Load tokenizer → tokenize dataset
2. Load model with `num_labels` set correctly
3. Define `compute_metrics`
4. Train with `Trainer`

> **Key difference from LLM fine-tuning:** we use
> `AutoModelForSequenceClassification`, not `AutoModelForCausalLM`.
> The model outputs a fixed-size logit vector (one per class),
> not a probability distribution over the vocabulary.

### A1. Tokenizer

> **Gotcha #1 — No pad token issue here, but padding side matters.**
> BERT/DistilBERT tokenizers already have a `[PAD]` token. Unlike GPT-style
> models you do not need to set one manually. However, set `truncation=True`
> and `max_length` when tokenizing or longer texts will raise an error.

In [ ]:
from transformers import AutoTokenizer

TEXT_MODEL = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
print(f'Loaded tokenizer: {TEXT_MODEL}')
print(f'Vocab size: {tokenizer.vocab_size:,}')
print(f'pad_token: {tokenizer.pad_token!r}  (already set — no fix needed)')

### A2. Dataset

We use a small inline dataset so the notebook runs without downloads.
In practice, swap this with `load_dataset('rotten_tomatoes')` or any
HuggingFace text classification dataset.

> **Gotcha #2 — Labels must be integers, not strings.**
> `Trainer` expects a `labels` column of type `int64`. If your dataset
> has string labels (`'positive'`, `'negative'`), you must encode them
> to integers and store the mapping in `id2label` / `label2id`.

In [ ]:
from datasets import Dataset
import numpy as np

# label: 0 = negative, 1 = positive
raw_text = [
    {'text': 'This film was absolutely brilliant, a masterpiece!', 'label': 1},
    {'text': 'Terrible acting, boring plot — a complete waste of time.', 'label': 0},
    {'text': 'A heartwarming story with stunning cinematography.', 'label': 1},
    {'text': 'The script was dreadful and the pacing unbearable.', 'label': 0},
    {'text': 'Loved every minute — highly recommend to everyone.', 'label': 1},
    {'text': 'Predictable, clichéd, and utterly forgettable.', 'label': 0},
    {'text': 'Outstanding performances and a gripping narrative.', 'label': 1},
    {'text': 'One of the worst films I have seen in years.', 'label': 0},
    {'text': 'A delightful surprise — clever writing and great chemistry.', 'label': 1},
    {'text': 'Dull and lifeless from start to finish.', 'label': 0},
    {'text': 'Exceptional direction and a moving conclusion.', 'label': 1},
    {'text': 'Poorly edited with too many unnecessary subplots.', 'label': 0},
]

id2label = {0: 'negative', 1: 'positive'}
label2id = {'negative': 0, 'positive': 1}

full_ds = Dataset.from_list(raw_text)
split = full_ds.train_test_split(test_size=0.25, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=128,
        padding=False,   # DataCollatorWithPadding handles dynamic padding
    )

train_tok = train_ds.map(tokenize, batched=True)
eval_tok  = eval_ds.map(tokenize,  batched=True)

# Trainer expects a column named 'labels', not 'label'
train_tok = train_tok.rename_column('label', 'labels')
eval_tok  = eval_tok.rename_column('label', 'labels')

train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_tok.set_format('torch',  columns=['input_ids', 'attention_mask', 'labels'])
print(train_tok.features)

### A3. Model

> **Gotcha #3 — Set `num_labels` at load time.**
> If you load `AutoModelForSequenceClassification` without `num_labels`,
> it defaults to 2 for binary tasks — which happens to be correct here,
> but for multi-class problems (3+ classes) the default is wrong and you
> will get a silent shape mismatch in the loss.
> Always pass `num_labels` explicitly.

> **Gotcha #4 — Expect a warning about the classification head.**
> Pre-trained DistilBERT has no classification head. When you load it with
> `AutoModelForSequenceClassification`, a new randomly-initialised head is
> attached. HuggingFace warns: *"Some weights were not used... Some weights
> were randomly initialized"*. This is **expected and correct** — it means
> the encoder is loaded from the checkpoint and the new head will be trained.

In [ ]:
from transformers import AutoModelForSequenceClassification

text_model = AutoModelForSequenceClassification.from_pretrained(
    TEXT_MODEL,
    num_labels=len(id2label),   # must set explicitly
    id2label=id2label,
    label2id=label2id,
)
params = sum(p.numel() for p in text_model.parameters())
trainable = sum(p.numel() for p in text_model.parameters() if p.requires_grad)
print(f'Total params: {params/1e6:.1f}M  Trainable: {trainable/1e6:.1f}M')
# All params are trainable — full fine-tuning (no LoRA needed at this scale)

### A4. Metrics and Training

> **Gotcha #5 — `compute_metrics` must return a dict.**
> A common mistake is returning a scalar (e.g., just `accuracy.compute()`).
> `Trainer` requires a `dict` mapping metric names to values. Returning
> anything else raises a cryptic `TypeError`.

> **Gotcha #6 — Use `DataCollatorWithPadding`, not the default collator.**
> The default `Trainer` collator stacks tensors of equal length. Text inputs
> are variable-length; `DataCollatorWithPadding` pads each batch dynamically
> to the longest sequence in that batch (more memory-efficient than static
> padding to `max_length`).

In [ ]:
import evaluate as ev
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

accuracy = ev.load('accuracy')
f1_metric = ev.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=preds, references=labels, average='binary')['f1'],
    }

training_args = TrainingArguments(
    output_dir='./distilbert_sentiment_output',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,         # lower than LLM fine-tuning; encoder is sensitive
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    report_to='none',
)

trainer_text = Trainer(
    model=text_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),  # dynamic padding
    compute_metrics=compute_metrics,
)

print('Training text classifier...')
trainer_text.train()
print('Evaluation:')
print(trainer_text.evaluate())

### A5. Inference

At inference, pass text through the tokenizer and model, then take the
`argmax` of the logits to get the predicted class.

In [ ]:
text_model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
text_model.to(device)

test_sentences = [
    'Absolutely fantastic — I laughed, I cried, I loved it.',
    'Slow, confusing, and not worth the ticket price.',
]
inputs = tokenizer(test_sentences, return_tensors='pt', truncation=True,
                   max_length=128, padding=True).to(device)
with torch.no_grad():
    logits = text_model(**inputs).logits
preds = logits.argmax(dim=-1)
for sent, pred in zip(test_sentences, preds):
    print(f'[{id2label[pred.item()].upper()}] {sent}')

In [ ]:
del text_model, trainer_text
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

---
## Section B — Image Classification with ViT

**Task:** plant disease classification using the `beans` dataset
(3 classes: angular leaf spot, bean rust, healthy — 1,034 training images)

Vision Transformers (ViT) split images into fixed-size patches and process
them like tokens. Fine-tuning follows the same `Trainer` pattern as BERT,
but uses `AutoImageProcessor` instead of a tokenizer.

### B1. Image Processor

> **Gotcha #7 — Use `AutoImageProcessor`, not a tokenizer.**
> Image models need an `AutoImageProcessor` (not `AutoTokenizer`) to
> resize, normalise, and convert images to tensors. The processor knows
> the right input size and normalisation stats for the pre-trained model.
> Using raw PIL images or a manual transform risks silent quality loss if
> the normalisation constants don't match the pre-training distribution.

In [ ]:
from transformers import AutoImageProcessor

VIT_MODEL = 'google/vit-base-patch16-224-in21k'
processor = AutoImageProcessor.from_pretrained(VIT_MODEL)
print(f'Image size : {processor.size}')
print(f'Mean       : {processor.image_mean}')
print(f'Std        : {processor.image_std}')

### B2. Dataset

The `beans` dataset is on the HuggingFace Hub — 1,034 train / 128 validation
images across 3 classes. It downloads automatically (~100 MB).

In [ ]:
from datasets import load_dataset

beans = load_dataset('beans')
print(beans)
print(f'Classes: {beans["train"].features["labels"].names}')

img_id2label = {i: name for i, name in enumerate(beans['train'].features['labels'].names)}
img_label2id = {v: k for k, v in img_id2label.items()}
print(f'id2label: {img_id2label}')

### B3. Preprocessing

> **Gotcha #8 — The dataset column is named `image` (PIL object), not a path.**
> HuggingFace image datasets store actual `PIL.Image` objects in the `image`
> column. Pass them directly to the processor — do **not** try to open them
> with `PIL.Image.open()` again (they are already loaded).

In [ ]:
def preprocess(batch):
    # batch['image'] is a list of PIL.Image objects — pass directly
    pixel_values = processor(images=batch['image'], return_tensors='pt')['pixel_values']
    return {'pixel_values': pixel_values, 'labels': batch['labels']}

train_img = beans['train'].map(preprocess, batched=True, remove_columns=['image'])
eval_img  = beans['validation'].map(preprocess, batched=True, remove_columns=['image'])

train_img.set_format('torch')
eval_img.set_format('torch')
print(train_img.features)

### B4. Custom Collator for Images

> **Gotcha #9 — The default Trainer collator does not handle `pixel_values`.**
> Unlike text (where `DataCollatorWithPadding` handles variable lengths),
> images are all the same size after processing. A simple collator that
> stacks tensors is all that's needed — but you must provide it, or Trainer
> will fail trying to handle the `image` column.

In [ ]:
import torch
from typing import Any

def image_collator(batch: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels':       torch.tensor([x['labels'] for x in batch]),
    }

### B5. Model and Training

> **Gotcha #10 — `ignore_mismatched_sizes=True` when changing `num_labels`.**
> ViT pre-trained on ImageNet-21k has 21,843 output classes. We need 3.
> Without `ignore_mismatched_sizes=True`, loading raises a `RuntimeError`
> about tensor size mismatch in the classifier head.

In [ ]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer
import evaluate as ev
import numpy as np

img_model = AutoModelForImageClassification.from_pretrained(
    VIT_MODEL,
    num_labels=len(img_id2label),
    id2label=img_id2label,
    label2id=img_label2id,
    ignore_mismatched_sizes=True,   # head size changes from 21k to 3
)

accuracy = ev.load('accuracy')

def img_compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy']}

img_training_args = TrainingArguments(
    output_dir='./vit_beans_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,   # keep pixel_values — Trainer strips unknown cols by default
    logging_steps=10,
    report_to='none',
)

trainer_img = Trainer(
    model=img_model,
    args=img_training_args,
    train_dataset=train_img,
    eval_dataset=eval_img,
    data_collator=image_collator,
    compute_metrics=img_compute_metrics,
)

print('Training ViT image classifier...')
trainer_img.train()
print('Evaluation:')
print(trainer_img.evaluate())

### B6. Inference


In [ ]:
from PIL import Image

img_model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
img_model.to(device)

# Use a sample from the test split
sample = beans['test'][0]
pil_image = sample['image']
true_label = img_id2label[sample['labels']]

inputs = processor(images=pil_image, return_tensors='pt').to(device)
with torch.no_grad():
    logits = img_model(**inputs).logits
pred_label = img_id2label[logits.argmax(-1).item()]
print(f'True label : {true_label}')
print(f'Predicted  : {pred_label}')

## Summary

### Gotcha recap

| # | Gotcha | Fix |
|---|--------|-----|
| 1 | BERT pad token — actually fine | Already set; but always check |
| 2 | String labels → crash | Encode to `int64`; store `id2label` / `label2id` |
| 3 | Wrong `num_labels` | Always pass explicitly to `from_pretrained` |
| 4 | Warning about random head init | Expected — encoder loads, new head trains |
| 5 | `compute_metrics` returns scalar | Must return a `dict` |
| 6 | Default collator fails on text | Use `DataCollatorWithPadding` |
| 7 | Using tokenizer for images | Use `AutoImageProcessor` |
| 8 | Reopening PIL images | Dataset already holds PIL objects; pass directly |
| 9 | Default collator fails on images | Write a simple stacking collator |
| 10 | Head size mismatch on ViT | Pass `ignore_mismatched_sizes=True` |

### LLM vs Non-LLM fine-tuning comparison

| | LLM (SFT) | Encoder/ViT |
|-|-----------|-------------|
| Model size | 1B–70B+ | 50M–350M |
| Full fine-tuning | Rarely practical | Common |
| LoRA | Standard | Possible but unusual |
| Tokenizer/processor | `AutoTokenizer` | `AutoTokenizer` or `AutoImageProcessor` |
| Loss | Next-token CE | Cross-entropy over fixed classes |
| Metric | Perplexity / task-specific | Accuracy, F1 |

**Next steps:** try `bert-base-uncased` on `glue/sst2` for a larger text
dataset, add `LoraConfig` to only train the attention layers for speed,
or fine-tune `openai/whisper-small` for speech recognition using the same
`Trainer` pattern.